# Avazu Click Prediction : Data Prep

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
# samples made in 01_eda , every 32nd row of each file
train = pd.read_parquet('../../data/ctr/train_sample.parquet')
valid = pd.read_parquet('../../data/ctr/valid_sample.parquet')
test = pd.read_parquet('../../data/ctr/test_sample.parquet')
len(train) , len(valid) , len(test)

(1010725, 126341, 126341)

## 3. Machine Learning
### Split : the given train / valid / test files are used as they are.
### Train is for fitting , valid for choosing settings (threshold , C) , test is used once at the end.

### Feature engineering : hour (YYMMDDHH) becomes hour of day and weekday.
### Feature selection , dropping :
### id : a row key.
### device_id , device_ip : hundreds of thousands of levels , they identify users , not a kind of impression.
### C14 : 0.98 correlated with C17 , and every C14 has exactly one C17 , so C17 is kept (fewer levels).
### C1 : 0.89 correlated with device_type , it is a relabelled device_type.
### app_domain : every app_id has exactly one app_domain.

In [3]:
drop = ['id' , 'hour' , 'device_id' , 'device_ip' , 'C14' , 'C1' , 'app_domain']

def features(d):
    d = d.copy()
    d['hour_of_day'] = d['hour'].str[6:]
    d['weekday'] = pd.to_datetime(d['hour'] , format='%y%m%d%H').dt.day_name()
    return d.drop(columns=drop)

train , valid , test = features(train) , features(valid) , features(test)
cat_cols = [c for c in train.columns if c != 'click']
cat_cols

['banner_pos',
 'site_id',
 'site_domain',
 'site_category',
 'app_id',
 'app_category',
 'device_model',
 'device_type',
 'device_conn_type',
 'C15',
 'C16',
 'C17',
 'C18',
 'C19',
 'C20',
 'C21',
 'hour_of_day',
 'weekday']

### Rare levels : site_id , app_id and device_model have thousands of levels , most seen a handful of times.
### Levels in fewer than 0.1% of train rows become 'other'. The levels are picked from train only , so valid and test don't leak in.

In [4]:
keep = {}
for col in cat_cols:
    share = train[col].value_counts(normalize=True)
    keep[col] = share[share >= 0.001].index

for d in [train , valid , test]:
    for col in cat_cols:
        d[col] = d[col].where(d[col].isin(keep[col]) , 'other')

train[cat_cols].nunique()

banner_pos            4
site_id              69
site_domain          63
site_category        10
app_id               54
app_category          9
device_model        153
device_type           4
device_conn_type      4
C15                   5
C16                   6
C17                 152
C18                   4
C19                  44
C20                  64
C21                  45
hour_of_day          24
weekday               7
dtype: int64

### Encoding : one hot. The categories come from train , so all three splits get the same columns.
### drop_first=True removes one column per feature since it can be worked out from the others.

In [5]:
for col in cat_cols:
    levels = sorted(train[col].unique())
    for d in [train , valid , test]:
        d[col] = pd.Categorical(d[col] , categories=levels)

def encode(d):
    X = pd.get_dummies(d[cat_cols] , drop_first=True , dtype='float32')
    return X , d['click']

X_train , y_train = encode(train)
X_val , y_val = encode(valid)
X_test , y_test = encode(test)
X_train.shape , X_val.shape , X_test.shape

((1010725, 703), (126341, 703), (126341, 703))

### Scaling : standardising every column with train values only.
### A rare level is almost always 0 , so without this gradient descent barely moves its weight.

In [6]:
scaler = StandardScaler().fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train) , columns=X_train.columns , dtype='float32')
X_val = pd.DataFrame(scaler.transform(X_val) , columns=X_train.columns , dtype='float32')
X_test = pd.DataFrame(scaler.transform(X_test) , columns=X_train.columns , dtype='float32')
X_train.iloc[: , :5].describe().round(2)

,banner_pos_1,banner_pos_7,banner_pos_other,site_id_0eb72673,site_id_12fb4121
count,1010725.00,1010725.00,1010725.00,1010725.00,1010725.00
mean,0.00,-0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-0.62,-0.03,-0.03,-0.05,-0.07
25%,-0.62,-0.03,-0.03,-0.05,-0.07
50%,-0.62,-0.03,-0.03,-0.05,-0.07
75%,1.61,-0.03,-0.03,-0.05,-0.07
max,1.61,29.81,38.46,19.62,14.33


In [7]:
X_train.to_parquet('../../data/ctr/X_train.parquet')
X_val.to_parquet('../../data/ctr/X_val.parquet')
X_test.to_parquet('../../data/ctr/X_test.parquet')
y_train.to_frame().to_parquet('../../data/ctr/y_train.parquet')
y_val.to_frame().to_parquet('../../data/ctr/y_val.parquet')
y_test.to_frame().to_parquet('../../data/ctr/y_test.parquet')